# Titanic Dataset — Data Cleaning & Preprocessing

## Week 1 Assignment

This notebook documents the complete process of **data acquisition, exploration, cleaning, validation, outlier detection, and preprocessing** using the supplied Titanic dataset.

### Objectives
- Explore the dataset structure and quality
- Identify and handle missing values
- Check duplicate and erroneous records
- Detect and treat statistical outliers
- Standardize categorical/text values
- Prepare a model-ready dataset
- Save cleaned datasets for further analysis

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

## 1. Data Acquisition and Loading

In [ ]:
# Load the supplied Titanic CSV
file_path = "Titanic-Dataset-selected-columns.csv"
df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Rows and columns:", df.shape)
df.head()

## 2. Initial Exploration

In [ ]:
# Dataset information
df.info()

In [ ]:
# Statistical summary
df.describe(include="all").T

In [ ]:
# Column names
print("Columns:")
for col in df.columns:
    print("-", col)

In [ ]:
# Missing values and unique values
quality_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Missing Values": df.isna().sum(),
    "Missing %": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique()
})
quality_summary

## 3. Missing Value Analysis

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
print(missing[missing > 0])

In [ ]:
# Plot missing values
missing_nonzero = missing[missing > 0]

if not missing_nonzero.empty:
    plt.figure(figsize=(8, 5))
    plt.bar(missing_nonzero.index, missing_nonzero.values)
    plt.title("Missing Values by Column")
    plt.xlabel("Column")
    plt.ylabel("Number of Missing Values")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("No missing values found.")

### Handling missing Age values

The `Age` column contains missing observations. These values are filled using the **median Age within each Sex + Pclass group**. A global median is used only as a fallback. This keeps all passenger records while using a more meaningful estimate than a single overall value.

In [ ]:
# Create cleaned copy
cleaned_df = df.copy()

# Grouped median imputation
cleaned_df["Age"] = cleaned_df.groupby(["Sex", "Pclass"])["Age"].transform(
    lambda x: x.fillna(x.median())
)

# Fallback for any remaining missing Age values
cleaned_df["Age"] = cleaned_df["Age"].fillna(cleaned_df["Age"].median())

print("Missing Age values after imputation:", cleaned_df["Age"].isna().sum())

## 4. Duplicate Record Check

In [ ]:
duplicate_count = cleaned_df.duplicated().sum()
print("Exact duplicate rows:", duplicate_count)

if duplicate_count > 0:
    cleaned_df = cleaned_df.drop_duplicates().reset_index(drop=True)
    print("Duplicate rows removed.")
else:
    print("No exact duplicate rows found.")

## 5. Check for Invalid or Erroneous Values

In [ ]:
# Check numerical columns for negative values
numeric_cols = ["Age", "SibSp", "Parch", "Fare"]

for col in numeric_cols:
    print(f"{col}: minimum = {cleaned_df[col].min()}")

print("\nNegative values:")
print((cleaned_df[numeric_cols] < 0).sum())

In [ ]:
# Logical range checks
print("Age outside 0–100:", ((cleaned_df["Age"] < 0) | (cleaned_df["Age"] > 100)).sum())
print("Negative Fare:", (cleaned_df["Fare"] < 0).sum())
print("Negative SibSp:", (cleaned_df["SibSp"] < 0).sum())
print("Negative Parch:", (cleaned_df["Parch"] < 0).sum())

## 6. Text and Categorical Standardization

In [ ]:
# Remove extra spaces and standardize text
for col in ["Name", "Sex", "Ticket"]:
    cleaned_df[col] = cleaned_df[col].astype(str).str.strip()

cleaned_df["Sex"] = cleaned_df["Sex"].str.lower()

print("Sex categories:", cleaned_df["Sex"].unique())

## 7. Outlier Detection Using IQR

In [ ]:
def iqr_outlier_summary(data, column):
    q1 = data[column].quantile(0.25)
    q3 = data[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (data[column] < lower) | (data[column] > upper)

    return {
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outlier Count": int(mask.sum())
    }

outlier_results = pd.DataFrame({
    col: iqr_outlier_summary(cleaned_df, col)
    for col in ["Age", "SibSp", "Parch", "Fare"]
}).T

outlier_results

In [ ]:
# Visualize Fare distribution before treatment
plt.figure(figsize=(8, 5))
plt.boxplot(cleaned_df["Fare"].dropna())
plt.title("Fare Outlier Detection")
plt.ylabel("Fare")
plt.tight_layout()
plt.show()

### Outlier treatment

An IQR outlier is statistically unusual but is not necessarily an incorrect observation. Therefore, valid passenger records are not deleted simply because they are unusual.

For `Fare`, extreme values are treated by **IQR capping** at the upper boundary. This reduces the influence of extreme values while preserving the rows.

In [ ]:
# Calculate Fare IQR limits
q1 = cleaned_df["Fare"].quantile(0.25)
q3 = cleaned_df["Fare"].quantile(0.75)
iqr = q3 - q1
fare_lower = q1 - 1.5 * iqr
fare_upper = q3 + 1.5 * iqr

print("Fare lower bound:", round(fare_lower, 2))
print("Fare upper bound:", round(fare_upper, 2))

# Cap Fare values
cleaned_df["Fare"] = cleaned_df["Fare"].clip(
    lower=fare_lower,
    upper=fare_upper
)

print("Fare outliers treated using IQR capping.")

## 8. Survival Distribution

In [ ]:
survival_counts = cleaned_df["Survived"].value_counts().sort_index()
print("Survival counts:")
print(survival_counts)

print(f"Overall survival rate: {cleaned_df['Survived'].mean() * 100:.2f}%")

In [ ]:
# Plot survival distribution
values = [
    (cleaned_df["Survived"] == 0).sum(),
    (cleaned_df["Survived"] == 1).sum()
]
labels = ["Did Not Survive", "Survived"]

plt.figure(figsize=(7, 5))
plt.bar(labels, values)
plt.title("Titanic Survival Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of Passengers")
plt.tight_layout()
plt.show()

## 9. Before vs After Cleaning

In [ ]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows", "Columns", "Missing Age Values",
        "Duplicate Rows", "Minimum Fare", "Maximum Fare"
    ],
    "Before Cleaning": [
        len(df), df.shape[1], df["Age"].isna().sum(),
        df.duplicated().sum(), df["Fare"].min(), df["Fare"].max()
    ],
    "After Cleaning": [
        len(cleaned_df), cleaned_df.shape[1], cleaned_df["Age"].isna().sum(),
        cleaned_df.duplicated().sum(), cleaned_df["Fare"].min(), cleaned_df["Fare"].max()
    ]
})
comparison

## 10. Create Model-Ready Dataset

In [ ]:
model_ready = cleaned_df.copy()

# Encode Sex
model_ready["Sex"] = model_ready["Sex"].map({
    "male": 0,
    "female": 1
})

# Standardize numerical features
features_to_scale = ["Age", "Fare", "SibSp", "Parch"]

for col in features_to_scale:
    mean = model_ready[col].mean()
    std = model_ready[col].std()
    model_ready[col] = (model_ready[col] - mean) / std

model_ready.head()

### Machine-learning note

For an actual ML workflow, imputation and scaling parameters should be learned from the **training set only** and then applied to validation/test data. This avoids data leakage. The model-ready dataset in this assignment is intended as a demonstration of preprocessing.

In [ ]:
# Verify model-ready numerical columns
model_ready[["Sex", "Age", "Fare", "SibSp", "Parch"]].describe()

## 11. Save Output Files

In [ ]:
# Save cleaned and model-ready datasets
cleaned_output = "Titanic_cleaned.csv"
model_output = "Titanic_model_ready.csv"

cleaned_df.to_csv(cleaned_output, index=False)
model_ready.to_csv(model_output, index=False)

print("Saved:", cleaned_output)
print("Saved:", model_output)

## 12. Final Data Quality Check

In [ ]:
print("Final shape:", cleaned_df.shape)

print("\nMissing values:")
print(cleaned_df.isna().sum())

print("\nDuplicate rows:", cleaned_df.duplicated().sum())

print("\nData types:")
print(cleaned_df.dtypes)

## Conclusion

The Titanic dataset was successfully explored, cleaned, and preprocessed. Missing Age values were handled using grouped median imputation, duplicate records were checked, numerical values were validated, categorical/text values were standardized, and Fare outliers were treated using IQR capping.

The resulting cleaned dataset can be used for further exploratory analysis, while the model-ready dataset demonstrates categorical encoding and numerical scaling for machine-learning applications.